# 1. Built-in signatures: from a signature to cell lines to drugs

`anchorline` anchors a query (a signature, a patient sample or single-cell data) to the closest untreated AML cell
lines in the drug-perturbation atlas. It then ranks the drugs tested in those lines by how much they change the cells,
whether they reverse the query signature, and how far they move the cells along the stem → myeloid axis.

The workflow has two steps, each computed once (`al.tl`), and plot functions that only choose what to show (`al.pl`):

```
mt = al.tl.match(atlas, query)              # rank all cell lines   -> al.pl.match_ranking, al.pl.pseudotime
dr = al.tl.rank_drugs(atlas, mt, cell_lines=5)   # rank drugs in them -> al.pl.drug_ranking, drug_shifts, state_map,
                                                 #                       class_trends, class_shift_map, gene_changes, ...
```
Every plot takes `n_top=` (how many) or explicit names (`cell_lines=`, `drugs=`, `classes=`), plus `figsize`,
`palette`, `cmap`, `fontsize`, `title`, `ax`, `save` and `show`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import anchorline as al

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
al.settings.figdir = "figures"          # save=True writes here
al.settings.set_figure_params(fontsize=8.5, dpi=150)
atlas = al.load()                       # first time: downloads the atlas (~560 MB) to al.settings.data_dir
atlas

## The signature library
`al.sig.list()` shows every bundled signature with its source. Collections:
- **Umeda2024_pediatric_AML**: 20 pediatric AML groups (Umeda et al. 2024 *Nat Genet* 56:281–293, doi:10.1038/s41588-023-01640-3), top 50 protein-coding genes per group, up and down.
- **Zeng2022_AML_hierarchy**: the AML cell-type signatures behind the pseudotime axis (Zeng et al. 2022 *Nat Med* 28:1212–1223, doi:10.1038/s41591-022-01819-x).
- **stemness**:
  - LSC104 (Ng et al. 2016 *Nature* 540:433–437, doi:10.1038/nature20598);
  - MLL-LSC (Somervaille et al. 2009 *Cell Stem Cell* 4:129–140, doi:10.1016/j.stem.2008.11.015);
  - HSC_Cycle_Primed (source not verified).
- **menin_targets**: the HOXA/MEIS1 core (Issa et al. 2023 *Nature* 615:920–924, doi:10.1038/s41586-023-05812-3).

In [ ]:
al.sig.collections()

In [ ]:
al.sig.list("Umeda2024")

In [ ]:
display(al.sig.search("MEIS1"))         # which signatures contain a gene
s = al.sig.get("Umeda2024_KMT2Ar_high")  # exact name, or a unique part of it: al.sig.get("KMT2Ar_high")
print(s); print("up:", s.up[:10], "...")

## Step 1: match the signature to cell lines
`al.tl.match` scores every untreated sample in the atlas and ranks all models. A cell line's score is the mean over its studies.
- `models="cell_lines"` is the default; `"patient_derived"` (PDX/primary only) and `"all"` are also available.
- Lymphoid lines are excluded unless you pass `exclude=()`.

In [ ]:
mt = al.tl.match(atlas, s)
mt

In [ ]:
mt.table.head(10)       # the full ranking is a DataFrame: filter it, export it, pick names from it

In [ ]:
al.pl.match_ranking(mt, n_top=15, highlight=5)

In [ ]:
al.pl.match_ranking(mt, by="study", n_top=15)     # cell line | study pairs

**Pseudotime.**
- **Top panel:** every untreated sample in the atlas (grey), with the most similar lines in colour.
- **Bottom panel:** only the matched lines, each untreated sample coloured by its own match score (viridis). The black tick marks the line's mean position.

In [ ]:
al.pl.pseudotime(mt, n_top=5)

## Step 2: rank the drugs tested in the chosen cell lines
Pass `cell_lines` as a number (the top n of the match) or as names, e.g. only the KMT2A-r lines.

In [ ]:
dr = al.tl.rank_drugs(atlas, mt, cell_lines=5)
dr

In [ ]:
dr.drugs.head(10)

In [ ]:
al.pl.drug_ranking(dr, n_top=15)                       # bars = score; markers = the three terms

In [ ]:
al.pl.drug_ranking(dr, n_top=20, color_by="drug_class", components=False)

In [ ]:
al.pl.drug_shifts(dr, n_top=12)                        # untreated -> treated pseudotime per drug

In [ ]:
al.pl.state_map(dr, n_top=6)                           # top 6 drugs ...
al.pl.state_map(dr, drugs=["ziftomenib", "pinometostat", "MI-3454", "SGC0946"], title="selected drugs")   # ... or by name

## Drug classes

In [ ]:
dr.classes.head(10)

In [ ]:
al.pl.class_trends(dr, n_top=12)

In [ ]:
al.pl.class_shift_map(dr, classes=["menin-KMT2A", "DOT1L", "BET", "DNMT/HMA", "HDAC", "BCL2"])

## Genes of interest
`gene_changes` shows any genes per drug (or class) × cell line. Colour is log2FC, size is the number of comparisons, and a hollow dot means the gene isn't expressed.

In [ ]:
al.pl.gene_changes(dr, genes=["CLEC12A", "CD84"], n_top=12)

In [ ]:
al.pl.gene_changes(dr, genes=["CLEC12A", "CD84", "CD33", "CD14"], classes=["menin-KMT2A", "DOT1L", "BET", "HDAC"],
                   title="selected genes by drug class")

In [ ]:
al.tl.gene_changes(dr, ["CLEC12A", "CD84"]).query("cell_line == 'pooled'").round(2)   # the numbers behind the plot

**Top gene changes per drug.**
- **A:** change indices. The net index is (genes up − genes down) / expressed genes; intensity is the mean |log2FC|.
- **B:** the most changed surface (SURFY) genes.
- **C:** the most changed protein-coding genes.

In [ ]:
al.pl.top_gene_changes(dr, n_top=12)

**The signature itself**: how each drug moves the query's up genes (orange labels) and down genes (blue labels). A reversal shows as blue up-genes and red down-genes, with a negative bar on the left.

In [ ]:
al.pl.signature_changes(dr, n_top=12)

In [ ]:
al.pl.summary(mt, dr, n_top=12, save=True)        # three-panel overview, saved to figures/

## Your choice of cell lines
After seeing the match, you can rank drugs in exactly the lines you trust. Here that is the four KMT2A-r lines, leaving out IMS-M2.

In [ ]:
dr_kmt2a = al.tl.rank_drugs(atlas, mt, cell_lines=["OCI-AML2", "NOMO-1", "MOLM-13", "MOLM-14"])
al.pl.drug_ranking(dr_kmt2a, n_top=12)

## Several signatures at once

In [ ]:
res = {}
for g in ["KMT2Ar_high", "HOXA", "RUNX1T1", "AMKL", "Immature", "NUP98r"]:
    m = al.tl.match(atlas, f"Umeda2024_{g}")
    res[g] = (m, al.tl.rank_drugs(atlas, m, cell_lines=5))
al.pl.group_summary(res, title="Umeda 2024 groups: matched lines, class shifts, top drugs")

In [ ]:
# export any table
mt.table.to_csv("figures/KMT2Ar_high_cell_lines.tsv", sep="\t")
dr.drugs.to_csv("figures/KMT2Ar_high_drugs.tsv", sep="\t")
al.tl.surface_genes(dr).to_csv("figures/KMT2Ar_high_surface_genes.tsv", sep="\t", index=False)